In [3]:
import networkx as nx
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

/home/pjgouze/anaconda3/envs/mini_kg_rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def build_kg():
    G = nx.DiGraph()

    # Exemple biomédical simplifié
    edges = [
        ("Tremor", "Parkinson", "symptom_of"),
        ("Rigidity", "Parkinson", "symptom_of"),
        ("Bradykinesia", "Parkinson", "symptom_of"),
        ("Parkinson", "Dopamine Deficiency", "related_to"),
        ("Dopamine Deficiency", "Basal Ganglia", "affects"),
        ("ALS", "Muscle Weakness", "symptom_of"),
        ("Stroke", "Paralysis", "symptom_of"),
    ]

    for u, v, rel in edges:
        G.add_edge(u, v, relation=rel)

    return G

In [5]:
def build_embeddings(G, model):
    node_list = list(G.nodes)
    embeddings = model.encode(node_list)
    embeddings = normalize(embeddings)

    node_to_idx = {node: i for i, node in enumerate(node_list)}
    idx_to_node = {i: node for node, i in node_to_idx.items()}

    return embeddings, node_to_idx, idx_to_node

In [6]:
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # produit scalaire (cosine si normalisé)
    index.add(embeddings.astype('float32'))
    return index

In [7]:
def search_nodes(index, query_embedding, k=5):
    query = query_embedding.reshape(1, -1).astype('float32')
    distances, indices = index.search(query, k)
    return indices[0]

In [8]:
def multi_hop_retrieval(G, query_embedding, index, embeddings, idx_to_node, hops=2, k=5):
    # Step 1: initial retrieval
    start_indices = search_nodes(index, query_embedding, k)
    current_nodes = [idx_to_node[i] for i in start_indices]

    visited = set(current_nodes)

    for _ in range(hops):
        candidates = set()

        # Expand neighbors
        for node in current_nodes:
            neighbors = list(G.neighbors(node))
            candidates.update(neighbors)

        candidates = list(candidates - visited)

        if not candidates:
            break

        # Score candidates
        candidate_embeddings = np.array([
            embeddings[list(idx_to_node.keys())[list(idx_to_node.values()).index(n)]]
            for n in candidates
        ])

        scores = np.dot(candidate_embeddings, query_embedding)

        # Select top-k
        top_k_idx = np.argsort(scores)[-k:]
        current_nodes = [candidates[i] for i in top_k_idx]

        visited.update(current_nodes)

    return list(visited)

In [9]:
def build_subgraph(G, nodes):
    return G.subgraph(nodes)

In [10]:
def linearize_graph(G_sub):
    triples = []
    for u, v, data in G_sub.edges(data=True):
        rel = data.get("relation", "related_to")
        triples.append(f"{u} --{rel}--> {v}")
    return "\n".join(triples)

In [11]:
def generate_answer(context, query):
    return f"""
Question: {query}

Knowledge Graph:
{context}

Answer:
Based on the graph, the most relevant disease is likely Parkinson,
as multiple symptoms connect to it.
"""

In [12]:
def run_pipeline(query):
    print("Building KG...")
    G = build_kg()

    print("Loading model...")
    model = SentenceTransformer("all-MiniLM-L6-v2")

    print("Building embeddings...")
    embeddings, node_to_idx, idx_to_node = build_embeddings(G, model)

    print("Building FAISS index...")
    index = build_faiss_index(embeddings)

    print("Encoding query...")
    query_embedding = model.encode([query])
    query_embedding = normalize(query_embedding)[0]

    print("Running multi-hop retrieval...")
    nodes = multi_hop_retrieval(
        G, query_embedding, index, embeddings, idx_to_node, hops=2, k=3
    )

    print("Retrieved nodes:", nodes)

    print("Building subgraph...")
    subgraph = build_subgraph(G, nodes)

    context = linearize_graph(subgraph)

    print("\nSubgraph:")
    print(context)

    answer = generate_answer(context, query)

    print("\nFinal Answer:")
    print(answer)